# PD Model – Paper-Grade Comparison (02a)

**Purpose:** Side-by-side comparison for a **credible paper**: **logistic regression** as the minimum interpretable baseline (practitioners and regulators compare against LR/scorecards), **feature importance stability** (SHAP ranking train vs OOT), and **all models with bootstrapped AUC confidence intervals**. Can feed into or replace parts of `02z_pd_model_comparison.ipynb` for the manuscript.

**Prerequisites:** Run `02a_pd_xgboost_training.ipynb` (saves `lr_baseline.pkl` and v2 stack) and `02b_pd_ann_training.ipynb` (saves ANN). For a full scorecard baseline (WOE + LR), add a separate training step or use a dedicated scorecard library.

## 1. Load data and model artifacts

In [ ]:
# Extends 02a_pd_xgboost_training.ipynb and 02z_pd_model_comparison.ipynb with publication-grade comparison
# LR baseline, SHAP stability (train vs OOT), bootstrapped AUC CIs
import sys
import json
import joblib
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names_no_leakage_v2
from credit_risk.feature_engineering.feature_screening import screen_features_train_only
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc, f1_score

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
MODEL_DIR = ROOT / "models" / "pd"
df = pd.read_parquet(DATA_PATH)
all_feature_names = get_feature_names_no_leakage_v2()
X = df[[c for c in all_feature_names if c in df.columns]].copy()
for c in all_feature_names:
    if c not in X.columns:
        X[c] = 0.0
X = X[all_feature_names]
y = df["default"]

train_idx = df["split"] == "train"
val_idx = df["split"] == "val"
test_idx = df["split"] == "test"
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

screening = screen_features_train_only(X_train, y_train, missingness_threshold=0.50, min_ks=0.001, corr_threshold=0.95)
feature_names = screening.selected_features
X_train = X_train[feature_names]
X_val = X_val[feature_names]
X_test = X_test[feature_names]
medians = X_train.median()
X_val_filled = X_val.fillna(medians)
X_test_filled = X_test.fillna(medians)

models = {}
if (MODEL_DIR / "pd_model_local_v2.pkl").exists():
    try:
        import credit_risk.models.pd_model
        models["XGBoost stack"] = joblib.load(MODEL_DIR / "pd_model_local_v2.pkl")
    except Exception:
        models["XGBoost stack"] = joblib.load(MODEL_DIR / "pd_model_local_v2.pkl")
if (MODEL_DIR / "pd_model_local_v1.pkl").exists() and "XGBoost stack" not in models:
    models["XGBoost stack"] = joblib.load(MODEL_DIR / "pd_model_local_v1.pkl")
if (MODEL_DIR / "pd_model_ann_v2.pkl").exists():
    models["ANN"] = joblib.load(MODEL_DIR / "pd_model_ann_v2.pkl")
if (MODEL_DIR / "lr_baseline.pkl").exists():
    models["Logistic Regression"] = joblib.load(MODEL_DIR / "lr_baseline.pkl")

print("Loaded:", list(models.keys()))

## 2. Test-set predictions and bootstrapped AUC CI per model

All models evaluated on the same OOT test set; report AUC with 95% bootstrap CI so the paper shows stability, not just a single number.

In [ ]:
def get_test_proba(name, data):
    if name == "XGBoost stack":
        return data["model"].predict_proba(X_test_filled)[:, 1]
    if name == "ANN":
        X_test_s = data["scaler"].transform(X_test_filled)
        return data["model"].predict_proba(X_test_s)[:, 1]
    # Logistic Regression
    X_test_s = data["scaler"].transform(X_test_filled)
    return data["model"].predict_proba(X_test_s)[:, 1]

N_BOOT = 1000
rng = np.random.default_rng(42)
n_test = len(y_test)
y_test_arr = np.asarray(y_test)

rows = []
preds_test = {}
for name, data in models.items():
    p_test = get_test_proba(name, data)
    preds_test[name] = p_test
    auc_boot = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n_test, size=n_test)
        auc_boot.append(roc_auc_score(y_test_arr[idx], p_test[idx]))
    auc_boot = np.array(auc_boot)
    ci_lo, ci_hi = np.percentile(auc_boot, [2.5, 97.5])
    point_auc = roc_auc_score(y_test, p_test)
    rows.append({"model": name, "test_auc": point_auc, "auc_ci_lo": ci_lo, "auc_ci_hi": ci_hi})

table_ci = pd.DataFrame(rows)
table_ci["AUC_95CI"] = table_ci.apply(lambda r: "{:.3f} [{:.3f}–{:.3f}]".format(r["test_auc"], r["auc_ci_lo"], r["auc_ci_hi"]), axis=1)
print("OOT test AUC with 95% bootstrap CI:")
print(table_ci[["model", "test_auc", "AUC_95CI"]].to_string(index=False))

## 3. Logistic regression baseline (interpretable / scorecard-style)

**LR is the minimum interpretable baseline** that credit practitioners and regulators compare against. Our `lr_baseline.pkl` is a standard L2 LR on the same screened features; it serves as the scorecard-style baseline. For a full **scorecard** (WOE binning + LR), you would add a separate training notebook or use a library (e.g. `scorecardpy`, `optbinning`). Here I only confirm LR is in the comparison table above.

In [ ]:
if "Logistic Regression" in models:
    lr_data = models["Logistic Regression"]
    p_lr = get_test_proba("Logistic Regression", lr_data)
    print("Logistic Regression OOT test AUC: {:.4f}".format(roc_auc_score(y_test, p_lr)))
    print("Coefficients (top 10 by abs value):")
    coef = pd.Series(lr_data["model"].coef_.ravel(), index=feature_names)
    coef = coef.iloc[np.argsort(np.abs(coef.values))[::-1]]
    print(coef.head(10).to_string())
else:
    print("Run 02a to train and save lr_baseline.pkl.")

## 4. Feature importance stability (SHAP: train vs OOT)

Compute SHAP importance (mean |SHAP|) on a **training sample** and on an **OOT test sample**. If the ranking is similar (e.g. high Spearman correlation), the same features drive predictions in both periods; if it shifts a lot, the model may be relying on different structure in OOT.

In [ ]:
if "XGBoost stack" in models:
    import shap
    data_xgb = models["XGBoost stack"]
    model_xgb = data_xgb["model"]
    model_for_shap = model_xgb.xgb_model if hasattr(model_xgb, "xgb_model") else model_xgb
    n_tr = min(2000, len(X_train))
    n_te = min(2000, len(X_test))
    X_tr_sample = X_train.fillna(medians).iloc[:n_tr]
    X_te_sample = X_test_filled.iloc[:n_te]
    explainer_tr = shap.TreeExplainer(model_for_shap, X_tr_sample)
    explainer_te = shap.TreeExplainer(model_for_shap, X_te_sample)
    sv_tr = explainer_tr.shap_values(X_tr_sample)
    sv_te = explainer_te.shap_values(X_te_sample)
    if isinstance(sv_tr, list):
        sv_tr = sv_tr[1] if len(sv_tr) > 1 else sv_tr[0]
    if isinstance(sv_te, list):
        sv_te = sv_te[1] if len(sv_te) > 1 else sv_te[0]
    imp_tr = np.abs(sv_tr).mean(axis=0)
    imp_te = np.abs(sv_te).mean(axis=0)
    rank_tr = np.argsort(np.argsort(-imp_tr))
    rank_te = np.argsort(np.argsort(-imp_te))
    from scipy.stats import spearmanr
    rho, pval = spearmanr(rank_tr, rank_te)
    print("SHAP importance stability (XGBoost): Spearman rho = {:.4f} (p = {:.4f})".format(rho, pval))
    imp_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap_train": imp_tr, "mean_abs_shap_test": imp_te})
    imp_df = imp_df.sort_values("mean_abs_shap_train", ascending=False)
    print("Top 10 by train SHAP:")
    print(imp_df.head(10).to_string(index=False))
else:
    print("XGBoost stack not loaded; skip SHAP stability.")

## 5. Summary table for the paper

All models side by side with OOT test AUC and 95% CI. Use this table (and the validation notebook’s temporal CV, calibration, PSI) in the manuscript.

In [ ]:
print(table_ci.to_string(index=False))
print("\nUse with paper_02a_validation.ipynb (temporal CV, calibration, PSI) for a complete validation story.")